In [4]:
import sys
sys.path.append('..')  # so we can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_onestop_corpus
from src.features import extract_all_features

# Display settings
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

In [5]:
corpus_path = r"../data/raw/OneStopEnglishCorpus/Texts-SeparatedByReadingLevel"
df = load_onestop_corpus(corpus_path)
df.head()

Loaded 567 texts across 3 levels.
Distribution:
level
Elementary      189
Intermediate    189
Advanced        189


,text,level,label,filename
0,"﻿When you see the word Amazon, what’s the first thing you think of – the world’s biggest forest,...",Elementary,0,Amazon-ele.txt
1,"﻿To tourists, Amsterdam still seems very liberal. Recently the city’s Mayor told them that the c...",Elementary,0,Amsterdam-ele.txt
2,"﻿Anitta, a music star from Brazil, has millions of fans, but she is at the centre of a debate ab...",Elementary,0,Anita-ele.txt
3,"Google has made maps of the world’s highest mountains, the ocean floor, the Amazon rainforest an...",Elementary,0,Arctic mapping-ele.txt
4,﻿The auction of a Banksy painting that disappeared from the wall of a north London shop was stop...,Elementary,0,Banksy-ele.txt


In [ ]:
print("Extracting features... this may take 5-10 minutes.")

features_list = []
errors = []

for idx, row in df.iterrows():
    try:
        features = extract_all_features(row['text'])
        features['label'] = row['label']
        features['level'] = row['level']
        features['filename'] = row['filename']
        features_list.append(features)
    except ValueError as e:
        errors.append((row['filename'], str(e)))

    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(df)} texts...")

features_df = pd.DataFrame(features_list)
print(f"\nDone! Extracted features for {len(features_df)} texts.")
print(f"Errors: {len(errors)}")

Extracting features... this may take 5-10 minutes.
  Processed 50/567 texts...
  Processed 100/567 texts...
  Processed 150/567 texts...
  Processed 200/567 texts...
  Processed 250/567 texts...
  Processed 300/567 texts...
  Processed 350/567 texts...
  Processed 400/567 texts...
  Processed 450/567 texts...
  Processed 500/567 texts...


In [ ]:
features_df.to_csv("../data/processed/features.csv", index=False)
print("Saved to data/processed/features.csv")

In [ ]:
feature_cols = [
    'lexical_density', 'nominalization_density',
    'clausal_complexity', 'mean_length_of_clause', 'passive_voice_ratio'
]

features_df.groupby('level')[feature_cols].mean().round(4)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    for level in ['Elementary', 'Intermediate', 'Advanced']:
        subset = features_df[features_df['level'] == level]
        ax.hist(subset[col], alpha=0.5, label=level, bins=20)
    ax.set_title(col.replace('_', ' ').title())
    ax.legend()

axes[5].set_visible(False)
plt.tight_layout()
plt.savefig("../data/processed/feature_distributions.png", dpi=150)
plt.show()